# 04 — Author-Level Topic Models (LDA) on the Two Backbones (HPC)

Fits author-level LDA topic models, once per network backbone written by
`04_Network_Analysis/01_network_analysis.ipynb`:

| Backbone | Community JSON (author set) | Authors |
|---|---|---:|
| `RetweetedOnce` — retweeted ≥ 1× by someone else, then LWCC | `Networks/4_communities/RetweetedOnce/Full_RetweetedOnce_author_communities.json` | 202,710 |
| `LWCC` — largest weakly connected component of the retweet graph | `Networks/4_communities/LWCC/Full_LWCC_author_communities.json` | 3,264,499 |

One **author document** = all of the author's tweets in the AI corpus
(`AItrust_twits_pruned_dict.json`: originals, replies, quotes and retweets), cleaned with
`clean_tweet_text()` and concatenated. A retweet contributes the retweeted text, so an author's
profile describes what they wrote *and* what they amplified. Authors with fewer than
`MIN_TWEETS_PER_AUTHOR` tweets are dropped.

**Per backbone:** (1) model selection on a seeded sample of authors — TF-IDF unigram / bigram ×
K × α × η, scored by coherence (c_v), topic diversity, collapse and evenness; (2) the selected
model refitted on **all** authors of the backbone; (3) one table per backbone with every author's
topic mixture and their five community labels. Section 6 adds author-level mean sentiment from
`01b`'s full AI output when that file is present.

**Before running on the cluster**
1. Copy both community JSONs from Drive (`Data Sets/Networks/4_communities/…`, written 2026-10-10)
   to the same relative path under `BASE_PATH` — the Setup cell checks for them.
2. Run once with `SMOKE_TEST = True` (minutes; writes to a separate `_smoke` folder), then with
   `SMOKE_TEST = False`.
3. Section 6 reads `ai_full_classified_twitter-roberta-base-sentiment-latest.json`; run it only
   after `01b` 8a has passed that file (trace §6.2.2). Sections 1–5 do not depend on it.

**History.** Replaces `04c_lda_author_topics_v2.ipynb` (deleted 2026-10-10; git history keeps it).
Every result stored in 04c was fitted on ~8,000 accounts selected by a GML-loader bug, so its
grids, chosen K, topic labels and UMAP/HDBSCAN experiments are not carried over; the settings that
the 04c exploration converged on are (TF-IDF, `max_df = 0.7`, `min_df = 5`, the v3 stopword list,
low α/η, the combined score of its §10.10). See `docs/AUTHOR_AND_NETWORK_PIPELINE_TRACE.md` §6.4.

# 0. Setup

In [ ]:
%%time
import os
from pathlib import Path

# Explorer / HPC project root (same as 01b)
BASE_PATH = Path("/projects/ComputationalPhilosophyLab/TwitterDataAnalysis")

cleanedds_folder    = BASE_PATH / "Data Sets" / "Cleaned Data"
networks_folder     = BASE_PATH / "Data Sets" / "Networks"
communities_folder  = networks_folder / "4_communities"      # 04_Network_Analysis/01's output
topic_models_folder = BASE_PATH / "Models" / "Topic Modeling"

TWEETS_PATH = cleanedds_folder / "AItrust_twits_pruned_dict.json"

# backbone -> its community export (keys = author ids as strings, values = {method: community})
BACKBONES = {
    "RetweetedOnce": communities_folder / "RetweetedOnce" / "Full_RetweetedOnce_author_communities.json",
    "LWCC":          communities_folder / "LWCC" / "Full_LWCC_author_communities.json",
}

SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
SENTIMENT_PATH  = cleanedds_folder / f"ai_full_classified_{SENTIMENT_MODEL.split('/')[-1]}.json"

missing = [p for p in [TWEETS_PATH, *BACKBONES.values()] if not p.exists()]
for p in [TWEETS_PATH, *BACKBONES.values(), SENTIMENT_PATH]:
    print(f"{'OK     ' if p.exists() else 'MISSING'}  {p}")
assert not missing, ("Missing inputs (the community JSONs are on Drive under "
                     "'Data Sets/Networks/4_communities/'; copy them to the same relative path): "
                     + ", ".join(str(p) for p in missing))

In [ ]:
%%time
import gc
import itertools
import json
import pickle
import random
import re
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.corpora import Dictionary as GensimDictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from tqdm.auto import tqdm

import sklearn, gensim
print("sklearn", sklearn.__version__, "| gensim", gensim.__version__)

# 1. Parameters

In [ ]:
%%time
SMOKE_TEST = False   # True: first 500,000 tweets, 5,000-author grid sample, one grid point -> OUT_DIR + "_smoke"

MIN_TWEETS_PER_AUTHOR = 3          # authors with fewer tweets in the AI corpus get no document
GRID_SAMPLE_AUTHORS   = 100_000    # authors (per backbone, seed 42) used for model selection
COHERENCE_DOCS        = 20_000     # documents of that sample used for c_v coherence
UMAP_SAMPLE_AUTHORS   = 50_000     # authors drawn for the UMAP plot of the final model

# Model-selection grid (04c's focused / higher-K grids narrowed to the region they favoured)
REPRESENTATIONS = {"tfidf_unigram": (1, 1), "tfidf_bigram": (1, 2)}
K_GRID      = [8, 12, 16, 20]
ALPHA_GRID  = [0.05, 0.1, 0.3]     # doc_topic_prior
ETA_GRID    = [0.01, 0.1]          # topic_word_prior
MAX_DF, MIN_DF = 0.7, 5
MAX_ITER    = 50
N_TOP_TERMS = 15                   # top terms per topic for coherence / diversity / collapse
SCORE_WEIGHTS = {"coherence": 0.25, "diversity": 0.30, "no_collapse": 0.15, "evenness": 0.30}  # 04c §10.10

# Override the automatic choice per backbone, e.g. {"LWCC": ("tfidf_unigram", 12, 0.3, 0.01)}
SELECTED_MODEL = {}

COMMUNITY_METHODS = ["label_propagation", "louvain", "leiden_fast", "leiden_directed", "infomap"]
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))
SEED = 42

MAX_TWEETS = None
if SMOKE_TEST:
    MAX_TWEETS, GRID_SAMPLE_AUTHORS, COHERENCE_DOCS, UMAP_SAMPLE_AUTHORS = 500_000, 5_000, 2_000, 5_000
    K_GRID, ALPHA_GRID, ETA_GRID, MAX_ITER = [8], [0.1], [0.1], 10

OUT_DIR = topic_models_folder / "LDA" / "author_level" / ("backbones_2026-10" + ("_smoke" if SMOKE_TEST else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR, "| N_JOBS:", N_JOBS, "| SMOKE_TEST:", SMOKE_TEST)
n_grid = len(REPRESENTATIONS) * len(K_GRID) * len(ALPHA_GRID) * len(ETA_GRID)
print(f"Grid: {n_grid} models per backbone, {len(BACKBONES)} backbones")

In [ ]:
%%time
# v3 stopword list of 04c §13 (the last one the 04c exploration used): sklearn English plus
# tweet artefacts, corpus-wide domain terms (every document is about AI) and filler words.
STOPWORDS = sorted(set(ENGLISH_STOP_WORDS) | {
    'http','https','user','rt','amp','tco','www','co',
    'ai','chatgpt','gpt','openai','google','microsoft',
    'artificial','intelligence','machine','learning',
    'chat','bot','model','data','technology','tech',
    'latest','search','bing','chatbot','generative','llm','deeplearning',
    'like','just','don','know','think','people','make','going',
    'want','really','good','time','thing','say','way','look',
    'right','come','got','need','let','get','go','take',
    'see','tell','give','try','ask','feel','talk','keep',
    'start','lot','much','still','even','would','could',
    'one','also','well','back','day','year','new','said',
    'may','actually','thanks','use','work','write','read',
    'help','learn','build','create','tool','app','world','today','week',
    'lol','oh','yeah','yes','ok',
    'gonna','wanna','gotta','didnt','doesnt','dont','cant','wont',
    'im','ive','id','youre','hes','shes','theyre','thats',
    'theres','whats','whos','hows',
})
print(f"{len(STOPWORDS)} stopwords")

# 2. Author documents

In [ ]:
%%time
def clean_tweet_text(text):
    # Same rules as preprocess() in 02_Processing/02: lowercase, drop the RT marker, URLs and
    # @mentions, keep hashtag words, collapse whitespace.
    text = (text or "").lower()
    text = re.sub(r'^rt\s+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+:?\s*', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = text.replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

for t in ["RT @AIartist: Check out my new #AIart piece! https://t.co/abc123",
          "@user123: I think #ChatGPT is going to revolutionize #creativity https://bit.ly/xyz"]:
    print(f"{t!r}\n  -> {clean_tweet_text(t)!r}")
assert clean_tweet_text("RT @a: x #AIart https://t.co/1") == "x aiart"

In [ ]:
%%time
# Author sets from the community exports. Keys are author ids as strings -- the same form as
# str(tweet["author_id"]). (Never take ids from a GML's `id` field: that is a row index; trace §6.4.)
COMMUNITIES = {}
for name, path in BACKBONES.items():
    with open(path, encoding="utf-8") as f:
        COMMUNITIES[name] = json.load(f)
    some = next(iter(COMMUNITIES[name].values()))
    print(f"{name:>13}: {len(COMMUNITIES[name]):>9,} authors, methods {sorted(some)}")
    assert set(COMMUNITY_METHODS) <= set(some), f"{name}: community methods missing"

AUTHOR_SETS = {name: set(c) for name, c in COMMUNITIES.items()}
ALL_AUTHORS = set().union(*AUTHOR_SETS.values())
inside = len(AUTHOR_SETS["RetweetedOnce"] & AUTHOR_SETS["LWCC"])
print(f"RetweetedOnce authors also in LWCC: {inside:,} of {len(AUTHOR_SETS['RetweetedOnce']):,}")
print(f"Authors to collect tweets for: {len(ALL_AUTHORS):,}")

In [ ]:
%%time
# One pass over the AI corpus collects the cleaned tweets of every backbone author.
# Cached to OUT_DIR so a restarted job does not stream the 17.4M tweets again.
CACHE = OUT_DIR / "author_texts.pkl"

if CACHE.exists():
    with open(CACHE, "rb") as f:
        author_texts = pickle.load(f)
    print(f"Loaded cached author texts: {len(author_texts):,} authors from {CACHE}")
else:
    author_texts = defaultdict(list)
    n_total = n_kept = n_empty = 0
    t0 = time.time()
    with open(TWEETS_PATH, encoding="utf-8") as f:
        for line in tqdm(itertools.islice(f, MAX_TWEETS), total=MAX_TWEETS or 17_410_035,
                         mininterval=60, desc="tweets"):
            if not line.strip():
                continue
            tw = json.loads(line)
            n_total += 1
            aid = str(tw.get("author_id", "")).strip()
            if aid not in ALL_AUTHORS:
                continue
            txt = clean_tweet_text(tw.get("text", ""))
            if not txt:
                n_empty += 1
                continue
            author_texts[aid].append(txt)
            n_kept += 1
    author_texts = dict(author_texts)
    print(f"Tweets read {n_total:,} | kept {n_kept:,} ({n_kept / max(n_total, 1):.1%}) | "
          f"empty after cleaning {n_empty:,} | authors with text {len(author_texts):,} | "
          f"{(time.time() - t0) / 60:.1f} min")
    if not SMOKE_TEST:
        assert n_total == 17_410_035, f"expected the 17,410,035 tweets of the AI corpus, read {n_total:,}"
        assert n_kept / n_total > 0.3, "far fewer backbone tweets than expected -- check the author-id format"
    with open(CACHE, "wb") as f:
        pickle.dump(author_texts, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("Cached to", CACHE)

In [ ]:
%%time
# Tweets per author, per backbone, then the threshold. Documents are built from the survivors.
author_docs = {}
for name, authors in AUTHOR_SETS.items():
    n = pd.Series({a: len(author_texts.get(a, ())) for a in authors}, dtype="int64")
    print(f"\n=== {name}: {len(n):,} authors, {n.sum():,} tweets ===")
    print(n.describe(percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())
    for k in (1, 2, 3, 5, 10, 20, 50):
        m = n >= k
        print(f"  >= {k:>3} tweets: {m.sum():>10,} authors ({m.mean():6.1%})  {n[m].sum():>12,} tweets")
    n.rename("n_tweets").rename_axis("author_id").to_csv(OUT_DIR / f"{name}_author_tweet_counts.csv")
    keep = sorted(a for a in authors if len(author_texts.get(a, ())) >= MIN_TWEETS_PER_AUTHOR)
    author_docs[name] = {a: " ".join(author_texts[a]) for a in keep}
    print(f"  -> {len(keep):,} author documents (>= {MIN_TWEETS_PER_AUTHOR} tweets)")
    assert len(keep) > 0

# 3. Model selection

For each backbone: a seeded sample of `GRID_SAMPLE_AUTHORS` documents, both representations,
every (K, α, η). Each model is scored on

- **coherence** — gensim c_v of the top terms on `COHERENCE_DOCS` sample documents, tokenised into
  unigrams for both representations (a bigram term counts as its two words), so the two are comparable;
- **diversity** — unique words / all words in the topics' top terms (Dieng et al. 2020);
- **collapse** — share of topic pairs sharing ≥ 70 % of their top-10 terms;
- **evenness** — `1 − max(0, largest dominant-topic share − 1/K)`, which penalises catch-all topics.

`combined_score` = the weighted sum of `SCORE_WEIGHTS` after min–max scaling coherence and
diversity within the backbone's grid. The best-scoring model is refitted in Section 4 unless
`SELECTED_MODEL` names another one.

In [ ]:
%%time
def top_terms(lda, vocab, n):
    return [[vocab[i] for i in comp.argsort()[::-1][:n]] for comp in lda.components_]

def diversity(tops):
    words = [w for t in tops for w in t]
    return len(set(words)) / len(words)

def collapse_ratio(tops, top_n=10, threshold=0.7):
    sets = [set(t[:top_n]) for t in tops]
    pairs = list(itertools.combinations(sets, 2))
    return sum(len(a & b) / top_n >= threshold for a, b in pairs) / len(pairs) if pairs else 0.0

def coherence_cv(tops, texts, dictionary):
    # Scored on unigram texts for both representations, so the two are comparable: a bigram
    # term contributes its two words (first occurrence kept, cut back to N_TOP_TERMS words).
    tops = [list(dict.fromkeys(w for term in t for w in term.split()))[:N_TOP_TERMS] for t in tops]
    known = [[w for w in t if w in dictionary.token2id] for t in tops]
    known = [t for t in known if len(t) >= 2]
    if not known:
        return np.nan
    cm = CoherenceModel(topics=known, texts=texts, dictionary=dictionary, coherence="c_v",
                        processes=max(N_JOBS - 1, 1))
    return float(cm.get_coherence())

def make_vectorizer(ngram_range):
    return TfidfVectorizer(lowercase=True, stop_words=STOPWORDS, max_df=MAX_DF, min_df=MIN_DF,
                           ngram_range=ngram_range, dtype=np.float32)

def make_lda(K, alpha, eta):
    return LatentDirichletAllocation(n_components=K, doc_topic_prior=alpha, topic_word_prior=eta,
                                     learning_method="batch", max_iter=MAX_ITER,
                                     random_state=SEED, n_jobs=N_JOBS)
print("metrics defined")

In [ ]:
%%time
GRIDS = {}
for name, docs in author_docs.items():
    ids = sorted(docs)
    if len(ids) > GRID_SAMPLE_AUTHORS:
        ids = sorted(random.Random(SEED).sample(ids, GRID_SAMPLE_AUTHORS))
    corpus = [docs[a] for a in ids]
    coh_idx = random.Random(SEED).sample(range(len(corpus)), min(COHERENCE_DOCS, len(corpus)))
    unigram = make_vectorizer((1, 1)).build_analyzer()          # lowercase + stopwords, no vocabulary
    texts = [unigram(corpus[i]) for i in coh_idx]
    dictionary = GensimDictionary(texts)
    print(f"\n=== {name}: grid on {len(corpus):,} of {len(docs):,} author documents ===")

    rows = []
    for rep, ngram in REPRESENTATIONS.items():
        vec = make_vectorizer(ngram)
        X = vec.fit_transform(corpus)
        vocab = vec.get_feature_names_out()
        print(f"  {rep}: vocabulary {len(vocab):,}, X {X.shape}")

        for K, alpha, eta in tqdm(list(itertools.product(K_GRID, ALPHA_GRID, ETA_GRID)), desc=f"{name}/{rep}"):
            t0 = time.time()
            lda = make_lda(K, alpha, eta).fit(X)
            tops = top_terms(lda, vocab, N_TOP_TERMS)
            dominant = lda.transform(X).argmax(axis=1)
            largest = np.bincount(dominant, minlength=K).max() / len(dominant)
            rows.append({
                "backbone": name, "representation": rep, "K": K, "alpha": alpha, "eta": eta,
                "coherence_c_v": coherence_cv(tops, texts, dictionary),
                "diversity": diversity(tops),
                "collapse_ratio": collapse_ratio(tops),
                "largest_topic_share": largest,
                "evenness": 1.0 - max(0.0, largest - 1.0 / K),
                "perplexity": float(lda.perplexity(X)),
                "n_iter": lda.n_iter_, "fit_minutes": (time.time() - t0) / 60,
            })
        del X, vec
        gc.collect()

    g = pd.DataFrame(rows)
    for col in ("coherence_c_v", "diversity"):
        lo, hi = g[col].min(), g[col].max()
        g[col + "_scaled"] = (g[col] - lo) / (hi - lo) if hi > lo else 0.5
    g["combined_score"] = (SCORE_WEIGHTS["coherence"] * g["coherence_c_v_scaled"].fillna(0)
                           + SCORE_WEIGHTS["diversity"] * g["diversity_scaled"]
                           + SCORE_WEIGHTS["no_collapse"] * (1 - g["collapse_ratio"])
                           + SCORE_WEIGHTS["evenness"] * g["evenness"])
    g = g.sort_values("combined_score", ascending=False).reset_index(drop=True)
    g.to_csv(OUT_DIR / f"{name}_grid_results.csv", index=False)
    GRIDS[name] = g
    print(g.head(10)[["representation", "K", "alpha", "eta", "coherence_c_v", "diversity",
                      "collapse_ratio", "largest_topic_share", "combined_score"]].round(3).to_string())

In [ ]:
%%time
fig, axes = plt.subplots(len(GRIDS), 4, figsize=(20, 4 * len(GRIDS)), squeeze=False)
for row, (name, g) in zip(axes, GRIDS.items()):
    for ax, col in zip(row, ["coherence_c_v", "diversity", "largest_topic_share", "combined_score"]):
        for rep, sub in g.groupby("representation"):
            best = sub.groupby("K")[col].max() if col != "largest_topic_share" else sub.groupby("K")[col].min()
            ax.plot(best.index, best.values, marker="o", label=rep)
        ax.set_title(f"{name}: {col} ({'min' if col == 'largest_topic_share' else 'max'} over alpha, eta)")
        ax.set_xlabel("K")
        ax.grid(alpha=.3)
    row[0].legend()
plt.tight_layout()
fig.savefig(OUT_DIR / "grid_quality_vs_K.png", dpi=150)
plt.show()

# 4. Final model per backbone

The selected configuration is refitted on **all** author documents of the backbone (vectorizer
and LDA). Written to `OUT_DIR/<backbone>/`:

- `vectorizer.pkl`, `lda_model.pkl`, `model.json` (configuration and fit statistics);
- `top_terms.csv` — 20 terms per topic with weights;
- `author_topics.csv` — one row per author: `author_id`, `n_tweets`, `topic_0 … topic_{K-1}`,
  `dominant_topic`, `dominant_weight`, and `community_<method>` for the five partitions of
  `04_Network_Analysis/01`.

In [ ]:
%%time
FINAL = {}
for name, docs in author_docs.items():
    rep, K, alpha, eta = SELECTED_MODEL.get(name) or tuple(
        GRIDS[name].loc[0, ["representation", "K", "alpha", "eta"]])
    K = int(K)
    out = OUT_DIR / name
    out.mkdir(exist_ok=True)
    ids = sorted(docs)
    print(f"\n=== {name}: {rep} K={K} alpha={alpha} eta={eta} on {len(ids):,} authors ===")

    t0 = time.time()
    vec = make_vectorizer(REPRESENTATIONS[rep])
    X = vec.fit_transform([docs[a] for a in ids])
    vocab = vec.get_feature_names_out()
    print(f"  vocabulary {len(vocab):,}, X {X.shape}, nnz {X.nnz:,}")
    lda = make_lda(K, alpha, eta).fit(X)
    theta = lda.transform(X).astype(np.float32)
    minutes = (time.time() - t0) / 60
    print(f"  fitted in {minutes:.1f} min ({lda.n_iter_} iterations)")

    with open(out / "vectorizer.pkl", "wb") as f:
        pickle.dump(vec, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(out / "lda_model.pkl", "wb") as f:
        pickle.dump(lda, f, protocol=pickle.HIGHEST_PROTOCOL)

    tops = top_terms(lda, vocab, 20)
    pd.DataFrame([{"topic": t, "rank": r, "term": vocab[i], "weight": float(comp[i])}
                  for t, comp in enumerate(lda.components_)
                  for r, i in enumerate(comp.argsort()[::-1][:20])]).to_csv(out / "top_terms.csv", index=False)

    df = pd.DataFrame(theta, columns=[f"topic_{t}" for t in range(K)])
    df.insert(0, "n_tweets", [len(author_texts[a]) for a in ids])
    df.insert(0, "author_id", ids)
    df["dominant_topic"] = theta.argmax(axis=1)
    df["dominant_weight"] = theta.max(axis=1)
    for m in COMMUNITY_METHODS:
        df[f"community_{m}"] = [int(COMMUNITIES[name][a][m]) for a in ids]
    df.to_csv(out / "author_topics.csv", index=False)

    shares = df["dominant_topic"].value_counts(normalize=True).sort_index()
    info = {"backbone": name, "representation": rep, "K": K, "alpha": float(alpha), "eta": float(eta),
            "max_df": MAX_DF, "min_df": MIN_DF, "max_iter": MAX_ITER, "n_iter": int(lda.n_iter_),
            "min_tweets_per_author": MIN_TWEETS_PER_AUTHOR, "n_authors": len(ids),
            "n_backbone_authors": len(AUTHOR_SETS[name]), "vocabulary": int(len(vocab)),
            "diversity": diversity([t[:N_TOP_TERMS] for t in tops]),
            "collapse_ratio": collapse_ratio(tops), "largest_topic_share": float(shares.max()),
            "fit_minutes": minutes, "smoke_test": SMOKE_TEST}
    (out / "model.json").write_text(json.dumps(info, indent=2))
    FINAL[name] = {"df": df, "tops": tops, "K": K, "rep": rep}

    print(f"  largest dominant-topic share {shares.max():.1%}")
    for t in range(K):
        print(f"  T{t:>2} ({shares.get(t, 0):5.1%}): {', '.join(tops[t][:10])}")
    del X, theta, vec, lda
    gc.collect()

In [ ]:
%%time
import umap

for name, r in FINAL.items():
    df, K = r["df"], r["K"]
    topic_cols = [f"topic_{t}" for t in range(K)]
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    sizes = df["dominant_topic"].value_counts().reindex(range(K), fill_value=0)
    axes[0].bar(range(K), sizes.values, color="steelblue")
    axes[0].set_xticks(range(K))
    axes[0].set_xticklabels([f"T{t}: {', '.join(r['tops'][t][:3])}" for t in range(K)], rotation=75, ha="right", fontsize=8)
    axes[0].set_ylabel("authors (dominant topic)")
    axes[0].set_title(f"{name}: {r['rep']} K={K} - {len(df):,} authors")

    sample = df.sample(min(UMAP_SAMPLE_AUTHORS, len(df)), random_state=SEED)
    emb = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(sample[topic_cols].values)
    sc = axes[1].scatter(emb[:, 0], emb[:, 1], c=sample["dominant_topic"], cmap="tab20", s=2, alpha=.5, rasterized=True)
    axes[1].set_title(f"{name}: UMAP of author topic mixtures ({len(sample):,} authors)")
    plt.colorbar(sc, ax=axes[1], label="dominant topic")
    plt.tight_layout()
    fig.savefig(OUT_DIR / name / "topic_sizes_and_umap.png", dpi=150)
    plt.show()

# 5. Checks

In [ ]:
%%time
for name, r in FINAL.items():
    out = OUT_DIR / name
    back = pd.read_csv(out / "author_topics.csv", dtype={"author_id": str})
    topic_cols = [c for c in back.columns if c.startswith("topic_")]
    assert len(back) == len(author_docs[name]), f"{name}: row count"
    assert back["author_id"].is_unique, f"{name}: duplicate authors"
    assert set(back["author_id"]) <= AUTHOR_SETS[name], f"{name}: author outside the backbone"
    assert (back["n_tweets"] >= MIN_TWEETS_PER_AUTHOR).all(), f"{name}: threshold not applied"
    assert np.allclose(back[topic_cols].sum(axis=1), 1, atol=1e-3), f"{name}: topic weights do not sum to 1"
    assert back[[f"community_{m}" for m in COMMUNITY_METHODS]].notna().all().all(), f"{name}: missing community"
    print(f"{name}: {len(back):,} authors x {len(topic_cols)} topics - checks passed "
          f"({len(back) / len(AUTHOR_SETS[name]):.1%} of the backbone's {len(AUTHOR_SETS[name]):,} authors)")

# 6. Author sentiment (needs `01b`'s full AI output)

Mean of the per-tweet sentiment scores of `cardiffnlp/twitter-roberta-base-sentiment-latest`
(`classifications[model]["scores"]` in `01b`'s output) over each backbone author's tweets.
Written to `OUT_DIR/author_sentiment_mean.csv` (`author_id`, `n_scored_tweets`, one column per
sentiment label), for every author of either backbone. Skipped with a message when the file is
not there; run it only after `01b` 8a has passed that file (trace §6.2.2).

In [ ]:
%%time
if not SENTIMENT_PATH.exists():
    print(f"Skipped: {SENTIMENT_PATH} not found (run 01b Section 8 first).")
else:
    sums = defaultdict(lambda: defaultdict(float))
    counts = defaultdict(int)
    labels = None
    with open(SENTIMENT_PATH, encoding="utf-8") as f:
        for line in tqdm(f, total=17_410_035, mininterval=60, desc="sentiment"):
            t = json.loads(line)
            aid = str(t.get("author_id", ""))
            if aid not in ALL_AUTHORS:
                continue
            scores = t.get("classifications", {}).get(SENTIMENT_MODEL, {}).get("scores")
            if not scores:
                continue
            labels = labels or sorted(scores)
            for lab in labels:
                sums[aid][lab] += scores[lab]
            counts[aid] += 1
    assert counts, f"no sentiment scores read from {SENTIMENT_PATH}"
    sent = pd.DataFrame([{"author_id": a, "n_scored_tweets": counts[a],
                          **{lab: sums[a][lab] / counts[a] for lab in labels}} for a in counts])
    sent.to_csv(OUT_DIR / "author_sentiment_mean.csv", index=False)
    for name, authors in AUTHOR_SETS.items():
        covered = sent["author_id"].isin(authors).sum()
        print(f"{name}: sentiment for {covered:,} of {len(authors):,} authors")
    print("Saved", OUT_DIR / "author_sentiment_mean.csv", f"({sum(counts.values()):,} tweets)")

In [ ]:
# Disconnect from Colab runtime (no-op on the cluster)
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Done:", OUT_DIR)